# 03 — Concept Leakage / Sufficiency  ·  Build step 04 (pre-G3) · Gate-G3 input · Novelty I2

Estimates **I(y ; features | concepts)** — the extra predictive information the encoder carries beyond the
clinical concepts — with honest cross-validated held-out log-likelihood. **Low** ⇒ concepts sufficient / faithful
(Path A). **High** ⇒ the model needs info outside the clinical concepts (Path B; the collapse/leakage is the finding).

Needs `concepts_all.npz` (nb01) and `M2_features.npy` (step 02). Output: `leakage_report.json`.

In [1]:
import sys, os
OWMTL_PKG = "/kaggle/input/datasets/barshonbasak/owmtl-package"
sys.path.insert(0, OWMTL_PKG)
import numpy as np, json
from owmtl.icbhi_data import KNOWN_DISEASES
OUT_DIR = "/kaggle/working"
Z = np.load("/kaggle/input/datasets/barshonbasak/concepts-all/concepts_all.npz", allow_pickle=True)
X = Z["X"].astype("float32"); patient = Z["patient"]; split = Z["split"]
diagnosis = Z["diagnosis"]; concept_names = list(Z["concept_names"])
F = np.load("/kaggle/input/datasets/barshonbasak/m2-features/M2_features.npy").astype("float32")
assert len(F) == len(X), f"M2 features ({len(F)}) are not aligned to the cycle index ({len(X)})"

# CYCLE level, not patient level. I(y;f|c) for a ~128-d f cannot be estimated from ~104
# patient rows: on synthetic data with a planted leak, the patient-level shape returned
# "LOW leakage / Path A" (see tests/test_leakage_and_split.py). Cycles give ~6900 rows;
# `groups` keeps the folds patient-independent so the estimate is not just patient identity.
lab_map = {d: i for i, d in enumerate(KNOWN_DISEASES)}
keep = np.array([d in lab_map for d in diagnosis])
Xc = X[keep]; Fc = F[keep]
yc = np.array([lab_map[d] for d in diagnosis[keep]])
groups = patient[keep]
print("cycles", len(yc), "| patients", len(set(groups.tolist())), "| classes", KNOWN_DISEASES)
print("class counts:", np.bincount(yc, minlength=len(KNOWN_DISEASES)))

cycles 6311 | patients 104 | classes ['COPD', 'Healthy', 'URTI']
class counts: [5746  322  243]


In [2]:
from owmtl.leakage import estimate_leakage
rep = estimate_leakage(yc, Xc, Fc, groups=groups, n_splits=5, seed=42)
print(json.dumps(rep, indent=2))
if not rep["estimator_valid"]:
    print("\n*** DO NOT read a G3 decision off this run -- see 'interpretation'. ***")
with open(os.path.join(OUT_DIR, "leakage_report.json"), "w") as fh: json.dump(rep, fh, indent=2)

{
  "leakage_bits": 0.2209,
  "leakage_frac_of_base": 0.1393,
  "logprob_bits": {
    "concepts_only": -0.5255,
    "concepts_plus_features": -0.3047,
    "features_only": -0.3025
  },
  "accuracy": {
    "concepts_only": 0.9092,
    "concepts_plus_features": 0.9212,
    "features_only": 0.9192
  },
  "accuracy_gain_from_features": 0.012,
  "uniform_baseline_bits": -1.585,
  "n_rows": 6311,
  "rows_per_feature": 8.1,
  "patient_grouped_cv": true,
  "estimator_valid": true,
  "interpretation": "MODERATE leakage: features add some signal beyond concepts; report and consider a leakage penalty.",
  "method": "held-out CV log-likelihood; leakage = I(y;f|c) in bits (arXiv:2504.09459 view)"
}


**Read `leakage_bits` / `interpretation`.** This is a Gate-G3 input: combine it with the accuracy tradeoff
(nb02) and the intervention effect (nb04) to decide Path A vs Path B. Do **not** pick the path here — that is the
G3 decision, made once all three numbers exist.